# 60 — Weak Bullet Rewriter
**Goal:** Selectively rewrite only low-scoring resume bullets using LLM.

## 1. Detection-First Strategy

In [ ]:
print('''Weak bullet rewriter pipeline:
1. Score each bullet using the STAR scorer (Notebook 37)
2. Only send weak bullets (score < 0.5) to LLM
3. Preserve strong bullets as-is
4. Review and accept/reject changes

Pros:
- Cost-effective (only rewrite ~40% of bullets)
- Preserves authentic writing
- Focuses LLM effort where it adds value''')

## 2. Bullet Scoring + Selective Rewrite

In [ ]:
import re

ACTION_VERBS = {"developed", "led", "reduced", "built", "designed", "implemented",
                "created", "managed", "delivered", "achieved", "improved"}

def score_bullet(bullet):
    """Score a bullet 0-1 based on quality signals."""
    text = bullet.strip()
    score = 0
    first_word = text.split()[0].lower().rstrip(",.;:")
    if first_word in ACTION_VERBS: score += 0.3
    if re.search(r"\\d+\\s*(%|million|billion|\\$)", text, re.IGNORECASE): score += 0.3
    if re.search(r"(using|with|via)\\s+[A-Z]", text): score += 0.15
    return min(score, 1.0)

bullets = [
    "Reduced model latency by 40% through TensorFlow optimization",
    "Was responsible for ML model development",
    "Worked on various data pipeline tasks",
]
for b in bullets:
    score = score_bullet(b)
    action = "SKIP (strong)" if score >= 0.5 else "→ SEND TO LLM"
    print(f"  {score:.2f} {action:16s}: {b[:50]}")

## 3. LLM Rewrite Prompt

In [ ]:
BULLET_REWRITE_PROMPT = """Rewrite the following resume bullet point to follow STAR format:
- Start with a strong action verb
- Include a quantified result where possible
- Be specific about technology and context

Original: {bullet}
Rewritten:"""

print("Prompt template ready.")
print("With API key, send to model and get rewritten version.")
print("\nExample rewrite:")
print("  Input:  'Was responsible for ML model development'")
print("  Output: 'Architected and deployed ML models achieving 95% accuracy, reducing manual review time by 40% using TensorFlow'")

## Summary: Selective rewriting cuts costs and preserves authentic writing. Only rewrite what's weak.